# Case study 2 - scoring the fine-tuned Cellpose 3 model

- **Does** - scores the model mycol fine-tuned for case study 2 against the curated ground-truth
  masks for the same 652 images (17,243 cells). Model and masks both come out of the saved session,
  so they cannot drift apart
- **Writes** - `../assets/cs2_cellpose3_results.csv`, the held-out counts Figure 3 panel **d** plots
- **Fair test** - scored only on the 20% mycol held out during fine-tuning (129 images, 3,341 cells).
  The split is reproduced here from `src/training/data_split.py` (`test_size=0.2`, `random_state=42`)
  and depends on `min_cells_per_image`, which is read off the session (5 here, dropping 11 sparse
  images) - so the held-out set is a property of the training run, not of the dataset
- **Metrics** - per-image count error (MAPE) and average precision at IoU 0.5 / 0.75 / 0.9. CS2 ships
  real masks, so AP is the more informative measure: it checks the cells are found in the right
  *places*, which a count can get right for the wrong reasons
- **Not here** - whether mycol should ship Cellpose 3 or 4 is answered in `cp3_vs_cp4/` on the `cp4`
  branch; the two versions cannot share an environment
- **Kernel** - `mycol_colonies_env` (cellpose 3.1.1.3), run top to bottom


In [ ]:
# --- locate the mycol repo root (so the app helpers import) and set paths ---
import json
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    """Walk up from `start` until the mycol source tree is found."""
    for p in [start, *start.parents]:
        if (p / "src" / "helpers" / "cellpose_functions.py").exists():
            return p
    raise RuntimeError("Could not locate the mycol repo root above %s" % start)


REPO_ROOT = find_repo_root(Path.cwd())
for extra in (REPO_ROOT, REPO_ROOT / "src" / "training"):   # src/training holds data_split.py
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

CS2_DATA = REPO_ROOT / "case_study_2"
SCRATCH = Path("output")            # this notebook's own scratch directory
SCRATCH.mkdir(exist_ok=True)
FIGURE_ASSETS = Path("..") / "assets"   # this figure's assets, one level up
SESSION_DIR = CS2_DATA / "mycol_saved_session_CS2"     # model + the curated images/masks
IMAGE_DIR = SESSION_DIR / "images"
MASK_DIR = SESSION_DIR / "masks"

MASK_SUFFIX = json.loads((SESSION_DIR / "image_metadata.json").read_text()).get("mask_suffix", "")

CP3_CSV = FIGURE_ASSETS / "cs2_cellpose3_results.csv"   # the counts Figure 3 panel d plots

print("repo root :", REPO_ROOT)
print("images    :", IMAGE_DIR, "(exists:", IMAGE_DIR.exists(), ")")
print("masks     :", MASK_DIR, "(exists:", MASK_DIR.exists(), f", suffix {MASK_SUFFIX!r})")


In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from data_split import split_train_test

MIN_CELLS = int(
    pd.read_csv(SESSION_DIR / "cellpose_training_hyperparameters.csv")
    .set_index("parameter")["value"]["min_cells_per_image"]
)

names = sorted(p.stem for p in IMAGE_DIR.glob("*.tif"))

upload_order = [str(n).removesuffix(".tif")
                for n in json.loads((SESSION_DIR / "image_metadata.json").read_text())["images"]]
assert upload_order == sorted(upload_order), "session upload order is not sorted name order"
assert set(upload_order) == set(names), "image_metadata.json and images/ disagree"


def load_mask(name: str) -> np.ndarray:
    """Curated ground-truth label image for one image (uint16, 0 = background)."""
    return np.array(Image.open(MASK_DIR / f"{name}{MASK_SUFFIX}.tif"))


true_counts = {n: int(np.unique(load_mask(n))[1:].size) for n in names}

kept = [n for n in names if true_counts[n] >= MIN_CELLS]
train_idx, test_idx = split_train_test(len(kept))          # test_size=0.2, seed=42
TRAIN = [kept[i] for i in train_idx]
TEST = [kept[i] for i in test_idx]
IS_TEST = {n: (n in set(TEST)) for n in names}

dropped = len(names) - len(kept)
print(f"images: {len(names)}   (min_cells_per_image = {MIN_CELLS}, "
      f"{dropped} dropped as too sparse to train on)")
print(f"  train split : {len(TRAIN):3}  ({sum(true_counts[n] for n in TRAIN):,} cells)")
print(f"  HELD OUT    : {len(TEST):3}  ({sum(true_counts[n] for n in TEST):,} cells)  <- the model never trained on these")
print(f"  total cells : {sum(true_counts.values()):,}")


In [ ]:
# --- load the fine-tuned model and the inference settings the app recorded for it ---
import torch
from cellpose import models as cp_models

MODEL_PATH = SESSION_DIR / "cellpose_model.pt"

hp = pd.read_csv(SESSION_DIR / "cellpose_inference_hyperparameters.csv")
stored = dict(zip(hp["parameter"].astype(str), hp["value"].astype(float)))
INFER = {
    "diameter": stored["diameter"] or None,        # the app writes 0 for "estimate automatically"
    "cellprob_threshold": stored["cellprob_threshold"],
    "flow_threshold": stored["flow_threshold"],
    "min_size": int(stored["min_size"]),
    "niter": int(stored["niter"]),
}

use_gpu = torch.cuda.is_available() or torch.backends.mps.is_available()
model = cp_models.CellposeModel(pretrained_model=str(MODEL_PATH), gpu=use_gpu)
print("fine-tuned Cellpose 3 model loaded (gpu=%s)" % use_gpu)
print("inference settings:", INFER)


In [ ]:
# --- inference: mirrors the app's predict_masks_with_cellpose pipeline exactly ---
from src.helpers.cellpose_functions import (
    preprocess_for_cellpose,
    convert_cellpose_mask_to_single_array,
)

BATCH = 8


def segment(names_list) -> dict:
    """Run the model over the given images; return {name: label image}."""
    out = {}
    for i in range(0, len(names_list), BATCH):
        chunk = names_list[i:i + BATCH]
        raws = [np.array(Image.open(IMAGE_DIR / f"{n}.tif").convert("RGB"), dtype=np.uint8)
                for n in chunk]
        ims = [preprocess_for_cellpose({"image": r}) for r in raws]   # grayscale + mean normalise
        masks, _, _ = model.eval(ims, channels=[0, 0], **INFER)
        for n, m, r in zip(chunk, masks, raws):
            H, W = r.shape[:2]
            out[n] = convert_cellpose_mask_to_single_array(m, H, W)
    return out


In [ ]:
# --- scoring: counts and segmentation quality against the curated masks ---
from cellpose import metrics as cp_metrics

AP_THRESHOLDS = [0.5, 0.75, 0.9]


def score(names_list, preds: dict) -> pd.DataFrame:
    """Per-image counts + average precision at several IoU thresholds.

    `average_precision` matches predicted to true objects by IoU, so AP@0.5 rewards getting the
    cells in the right places, not merely getting the total right - the counts alone can be correct
    for the wrong reasons (a missed cell cancelling a spurious one).
    """
    gt = [load_mask(n) for n in names_list]
    pr = [preds[n] for n in names_list]
    ap, tp, fp, fn = cp_metrics.average_precision(gt, pr, threshold=AP_THRESHOLDS)
    rows = []
    for i, n in enumerate(names_list):
        t = true_counts[n]
        p = int(np.unique(pr[i])[1:].size)
        rows.append({
            "image": n,
            "split": "test" if IS_TEST[n] else "train",
            "true_count": t,
            "pred_count": p,
            "abs_pct_error": abs(p - t) / t * 100,
            "signed_pct_error": (p - t) / t * 100,
            **{f"AP@{th}": ap[i, j] for j, th in enumerate(AP_THRESHOLDS)},
            "TP@0.5": int(tp[i, 0]), "FP@0.5": int(fp[i, 0]), "FN@0.5": int(fn[i, 0]),
        })
    return pd.DataFrame(rows)


def summarise(d: pd.DataFrame, label: str) -> pd.DataFrame:
    """One row per model, over the held-out test images."""
    tp, fp, fn = d["TP@0.5"].sum(), d["FP@0.5"].sum(), d["FN@0.5"].sum()
    return pd.DataFrame({f"{label}  (n={len(d)})": {
        "MAPE %": d["abs_pct_error"].mean(),
        "signed %": d["signed_pct_error"].mean(),
        "AP@0.5": d["AP@0.5"].mean(),
        "AP@0.75": d["AP@0.75"].mean(),
        "AP@0.9": d["AP@0.9"].mean(),
        "F1@0.5": (2 * tp) / (2 * tp + fp + fn),
    }}).T.round(4)


In [ ]:
import time

if CP3_CSV.exists():
    cp3 = pd.read_csv(CP3_CSV)
    print("loaded cached results:", CP3_CSV.name, "(delete it to recompute)")
else:
    t0 = time.time()
    preds = {}
    for i in range(0, len(TEST), 64):
        preds.update(segment(TEST[i:i + 64]))
        print(f"  segmented {min(i + 64, len(TEST))}/{len(TEST)}"
              f"   [{time.time() - t0:5.0f}s]", flush=True)
    cp3 = score(TEST, preds)
    cp3.to_csv(CP3_CSV, index=False)
    print(f"\nwrote {CP3_CSV.name}  ({time.time() - t0:.0f}s)")

cp3.head()


In [ ]:
# --- headline (held-out test images only) ---
summary = summarise(cp3, "Cellpose 3 (fine-tuned)")
print("Case study 2 - fine-tuned Cellpose 3, held-out test set\n")
summary


In [ ]:
def true_vs_pred(ax, true, pred, color, title, s=26, alpha=0.6):
    """True vs predicted per-image cell count, with the y = x reference."""
    true = np.asarray(true, dtype="float64")
    pred = np.asarray(pred, dtype="float64")
    ax.set_title(title, fontsize=11, fontweight="bold")
    lim = max(true.max(), pred.max()) * 1.05
    ax.plot([0, lim], [0, lim], color="#8a8a8a", lw=1.2, ls="--", zorder=1)
    ax.scatter(true, pred, s=s, alpha=alpha, color=color, zorder=2,
               edgecolors="white", linewidths=0.5)
    ax.set_xlim(-lim * 0.02, lim); ax.set_ylim(-lim * 0.02, lim); ax.set_aspect("equal")
    m = (np.abs(pred - true) / true * 100).mean()
    ax.text(0.05, 0.95, f"MAPE {m:.2f}%\nn = {len(true):,}", transform=ax.transAxes, va="top",
            ha="left", fontsize=9, bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#dddddd"))
    ax.text(lim * 0.97, lim * 0.9, "y = x", color="#8a8a8a", fontsize=8, rotation=45,
            rotation_mode="anchor", va="bottom", ha="right")
    ax.set_xlabel("true count"); ax.set_ylabel("predicted count")
    ax.grid(True, lw=0.4, color="#ededed"); ax.set_axisbelow(True)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)


In [ ]:
# --- true vs predicted count ---
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4.8, 4.8))
true_vs_pred(ax, cp3["true_count"], cp3["pred_count"], "#0072B2",
             "Cellpose 3 (fine-tuned) - held out")
fig.tight_layout()
fig.savefig(SCRATCH / "cs2_cellpose3_true_vs_pred.png", dpi=150, bbox_inches="tight",
            facecolor="white")
plt.show()
